# Lab 2: modelling the Adult dataset

Team E. This notebook does not contain the schema. The schema lives in the team repository under `sql/`, and this notebook clones that repository and runs it.
Repository: https://github.com/zukazudo/dsai6226-data-engineering

In [13]:
%cd /content
!rm -rf dsai6226-data-engineering
!git clone -q https://github.com/zukazudo/dsai6226-data-engineering.git
%cd /content/dsai6226-data-engineering
!pip install -q duckdb
!ls sql

/content
/content/dsai6226-data-engineering
01_staging.sql	02_schema.sql  03_load.sql  04_query.sql


**Observation**

The repository carries four SQL files that run in order. `01_staging.sql` lands the CSV as raw text, `02_schema.sql` creates the star, `03_load.sql` transforms staging into it, and `04_query.sql` asks the allocation question. Cleaning happens only in the third file, so the raw value behind every modelled row stays queryable.

The cell above deletes and re-clones on every run, so it can be re-executed safely.

In [14]:
!python scripts/build_warehouse.py

  01_staging.sql           211 ms
  02_schema.sql             35 ms
  03_load.sql              904 ms

  dim_education               16 rows
  dim_marital_status           7 rows
  dim_native_country          42 rows
  dim_occupation              16 rows
  dim_race                     5 rows
  dim_relationship             6 rows
  dim_sex                      2 rows
  dim_workclass                9 rows
  fact_person             32,561 rows
  stg_adult               32,561 rows

warehouse written to warehouse/adult.duckdb


**Observation**

Eight dimensions and one fact table, built in roughly two seconds. All 32,561 source rows reach `fact_person`, because the duplicate decision is to retain and mark rather than to drop.

Two dimension counts are worth reading. `dim_occupation` holds 16 rows for 14 real occupations, because it adds `Unknown` and `Not applicable`. `dim_workclass` and `dim_native_country` each add only `Unknown`.

In [15]:
import duckdb
import pandas as pd

con = duckdb.connect("warehouse/adult.duckdb", read_only=True)

tables = [r[0] for r in con.execute(
    "SELECT table_name FROM information_schema.tables WHERE table_schema = 'main' ORDER BY table_name"
).fetchall()]

for t in tables:
    n = con.execute("SELECT count(*) FROM " + t).fetchone()[0]
    print(t.ljust(22), format(n, ",").rjust(8), "rows")

print()
print("fact_person columns:")
print(con.execute("DESCRIBE fact_person").df()[["column_name", "column_type"]].to_string(index=False))

dim_education                16 rows
dim_marital_status            7 rows
dim_native_country           42 rows
dim_occupation               16 rows
dim_race                      5 rows
dim_relationship              6 rows
dim_sex                       2 rows
dim_workclass                 9 rows
fact_person              32,561 rows
stg_adult                32,561 rows

fact_person columns:
             column_name column_type
               person_sk      BIGINT
              source_row     INTEGER
            workclass_sk     INTEGER
            education_sk     INTEGER
       marital_status_sk     INTEGER
           occupation_sk     INTEGER
         relationship_sk     INTEGER
                 race_sk     INTEGER
                  sex_sk     INTEGER
       native_country_sk     INTEGER
                     age    SMALLINT
         age_is_topcoded     BOOLEAN
          hours_per_week    SMALLINT
       hours_is_topcoded     BOOLEAN
            capital_gain     INTEGER
capital_gain_is_

**Observation**

The fact table is narrow: eight foreign keys, six measures, and the bookkeeping columns that make the Lab 1 defects addressable.

- `person_sk` is the surrogate key Lab 1 found missing. It is the only reason a row here can be referred to, updated or joined at all.
- `source_row` traces any modelled row back to the exact line of the CSV it came from.
- The three `_is_topcoded` booleans record censoring that the source hid inside the values themselves.
- `duplicate_group_id` and `duplicate_seq` record the duplicate decision instead of applying it silently.

In [16]:
# The source wrote "?" for two different things. The dimension separates them.
print(con.execute("""
    SELECT o.occupation, o.is_unknown, o.is_not_applicable, count(*) AS records
    FROM fact_person f
    JOIN dim_occupation o USING (occupation_sk)
    WHERE o.is_unknown OR o.is_not_applicable
    GROUP BY 1, 2, 3
""").df().to_string(index=False))

print()
print("workclass of the Not applicable records, straight from staging:")
print(con.execute("""
    SELECT s.workclass, count(*) AS records
    FROM fact_person f
    JOIN stg_adult s ON s.source_row = f.source_row
    WHERE f.occupation_sk = -2
    GROUP BY 1
""").df().to_string(index=False))

    occupation  is_unknown  is_not_applicable  records
       Unknown        True              False     1836
Not applicable       False               True        7

workclass of the Not applicable records, straight from staging:
   workclass  records
Never-worked        7


**Observation**

1,836 records did not answer the occupation question and 7 have no occupation to record. The source file wrote `?` for both. All 7 of the second group are `Never-worked`, confirmed here by joining back to staging, which is what the `source_row` lineage column is for.

This is the clearest argument for the dimension. A flat table has only two options: keep the magic string `?`, which is the defect Lab 1 documented, or write NULL and destroy a real value for those 7 people.

In [17]:
# Run the committed query file rather than retyping the SQL here.
sql = open("sql/04_query.sql", encoding="utf-8").read()

def is_runnable(stmt):
    body = [l for l in stmt.splitlines() if l.strip() and not l.strip().startswith("--")]
    return bool(body)

stmts = [s.strip() for s in sql.split(";") if is_runnable(s)]
print("runnable statements in sql/04_query.sql:", len(stmts))

alloc = con.execute(stmts[0]).df()
print("segments:", len(alloc), " of which under 30 candidates:", int((alloc.candidates < 30).sum()))
print(alloc.head(12).to_string(index=False))

runnable statements in sql/04_query.sql: 4
segments: 80  of which under 30 candidates: 18
       occupation      education_group  candidates  mean_hours  censored_rows  mean_capital_gain note
     Craft-repair High school graduate        1197        42.3              0             157.44 None
     Adm-clerical High school graduate         795        39.4              0             143.47 None
    Other-service High school graduate         778        37.3              0              79.55 None
   Prof-specialty     Bachelors degree         712        42.2              0             192.76 None
     Adm-clerical         Some college         678        39.3              0             144.18 None
Machine-op-inspct High school graduate         664        41.1              0             200.81 None
 Transport-moving High school graduate         528        45.7              0              97.35 None
            Sales High school graduate         505        41.9              0             180.

**Observation**

This is the query the whole model exists to answer: how many eligible candidates sit in each occupation and education segment.

Lab 1 found 93 of 201 raw segments held fewer than 30 records, which is too thin to allocate a budget against. After `education_group` rolls the 16 education levels into 6, it is 18 of 80. The rollup lives in `dim_education`, so it is defined once and applies to every query rather than being retyped, slightly differently, into each one.

In [18]:
print(con.execute(stmts[1]).df().to_string(index=False))

                          measure  records
                    eligible pool    16005
of those, occupation not answered      519
of those, no occupation to record        1


**Observation**

Lab 1 reported 520 unassignable candidates in the eligible pool and had no way to break that number down. The model splits it: **519 did not answer** and **1 has never worked**.

Only the first group is a data collection problem. The second is a correctly recorded fact about a person. Reporting them as one number of 520 overstates the gap, and the schema is what makes the distinction available.

In [19]:
print(con.execute(stmts[2]).df().to_string(index=False))

       occupation  mean_excluding_sentinel  mean_including_sentinel  censored_rows
   Prof-specialty                   1127.0                   2727.0             67
  Exec-managerial                   1243.0                   2263.0             42
            Sales                    639.0                   1320.0             25
  Protective-serv                    555.0                    708.0              1
     Tech-support                    566.0                    674.0              1
     Craft-repair                    455.0                    650.0              8
  Farming-fishing                    590.0                    590.0              0
     Adm-clerical                    337.0                    496.0              6
 Transport-moving                    428.0                    490.0              1
Machine-op-inspct                    279.0                    329.0              1
  Priv-house-serv                    280.0                    280.0              0
Hand

**Observation**

Why `capital_gain` is stored NULL with a flag instead of as 99999. Ranking the 14 occupations by mean capital gain, 9 change position once the sentinel is excluded, and the top two swap.

`Farming-fishing` is the clearest case. It is the only occupation with no censored records at all, so its mean of 590 does not move, yet it still rises from seventh to fourth because the inflated occupations above it fall away. It does not move. Everything else moves around it.

A budget allocated on the uncorrected ranking would be allocated on an artifact of the encoding.

In [20]:
print(con.execute(stmts[3]).df().to_string(index=False))

 rows_loaded  rows_in_a_duplicate_group  duplicate_groups  rows_pandas_would_call_duplicated
       32561                         47                23                                 24


**Observation**

The duplicate decision, recorded rather than applied. All 32,561 rows load. 47 rows belong to one of 23 duplicate groups, and 24 of those are the rows `pandas.duplicated()` would flag, being every group member after the first.

Lab 1 concluded that dropping them is not safe, because a data entry duplicate and two genuinely similar people cannot be told apart without a key. A deduplicated view is one predicate away, `WHERE duplicate_seq IS NULL OR duplicate_seq = 1`, and the choice stays reversible.

## Why not one big table

For 32,561 rows, one big table would be faster and simpler, and DuckDB would scan it without noticing. The reason to model it anyway is not performance:

- A flat table has nowhere to put the difference between **missing** and **not applicable**. That distinction is the 7 `Never-worked` records above.
- A flat table cannot hold the **duplicate decision**, which needs columns describing the record rather than the person, sitting beside a surrogate key that a flat file does not have.
- `education` and `education.num` are **redundant in every one of the 32,561 rows**, and nothing stops them drifting. In `dim_education` the pairing is asserted once and enforced by a unique constraint.
- **Rollups belong in one place.** `education_group` is a decision about how to aggregate. Defined in a dimension it applies everywhere.

The honest counter-argument: `stg_adult` **is** one big table, and it is kept deliberately. It is the raw record, and the star is derived from it.